# 90 — Optional training source diagnostics
Run after 01–05. These detailed audits are evidence for the report, not required modelling steps. They do not alter labels or model exports. Suspected causes remain unconfirmed.

In [ ]:
import pandas as pd
import numpy as np
import json
import pricecatcher_config as cfg
from importlib import reload
cfg = reload(cfg)
print("Experiment:", cfg.EXPERIMENT, "| Training cutoff:", cfg.TRAIN_END_DATE)
# Exact-calendar feature builder
def make_features(one_series):
    """Build calendar-week history and audit outcomes for one local price series."""
    grid = one_series.set_index("week_start").sort_index()
    calendar = pd.date_range(grid.index.min(), grid.index.max(), freq="W-MON")
    grid = grid.reindex(calendar)
    price = grid["weekly_median_price"]

    for lag in [1, 2, 4]:
        grid[f"lag_{lag}w_price"] = price.shift(lag)
    for window in [4, 8]:
        past = price.shift(1)
        grid[f"rolling_mean_{window}w"] = past.rolling(window, min_periods=2).mean()
        grid[f"rolling_std_{window}w"] = past.rolling(window, min_periods=2).std()
    grid["history_count_4w"] = price.shift(1).rolling(4, min_periods=1).count()
    grid["weekly_price_change_pct"] = (price / price.shift(1) - 1) * 100

    # These FUTURE columns are for labels and audit only, never model features.
    grid["next_week_price"] = price.shift(-1)
    grid["next_week_record_count"] = grid["record_count"].shift(-1)
    grid["next_week_premise_count"] = grid["premise_count"].shift(-1)
    grid["next_week_change_pct"] = (grid["next_week_price"] / price - 1) * 100

    # Return original observed rows, not the inserted empty calendar rows.
    grid = grid.loc[price.notna()].copy()
    grid.index.name = "week_start"
    return grid.reset_index()

OUTPUT_DIR = cfg.STAGE5_DIR
TRAIN_END = pd.Timestamp(cfg.TRAIN_END_DATE)
SERIES_KEYS = cfg.GROUP_KEYS
MIN_RECORDS, MIN_PREMISES = cfg.MIN_RECORDS, cfg.MIN_PREMISES
MIN_HISTORY_4W = cfg.MIN_HISTORY_4W
settings = json.loads((OUTPUT_DIR / "experiment_settings.json").read_text())
if {key: settings["preparation"][key] for key in cfg.PREPARATION_SETTINGS} != cfg.PREPARATION_SETTINGS:
    raise ValueError("Settings changed. Rerun 02–05 before optional diagnostics.")
weekly = pd.read_parquet(cfg.STAGE4_WEEKLY_PARQUET)
weekly["week_start"] = pd.to_datetime(weekly["week_start"])
weekly["week_end"] = weekly["week_start"] + pd.Timedelta(days=6)
# Audits cannot use validation or test observations.
weekly = weekly.loc[weekly["week_end"] <= TRAIN_END].copy()
featured = pd.concat([make_features(part) for _, part in weekly.groupby(SERIES_KEYS, sort=False)], ignore_index=True)
featured["next_week_start"] = featured["week_start"] + pd.Timedelta(days=7)
featured["next_week_end"] = featured["week_start"] + pd.Timedelta(days=13)
reference_mask = (featured["record_count"].ge(MIN_RECORDS)
    & featured["premise_count"].ge(MIN_PREMISES)
    & featured["next_week_record_count"].ge(MIN_RECORDS)
    & featured["next_week_premise_count"].ge(MIN_PREMISES)
    & featured["next_week_price"].notna() & featured["next_week_end"].le(TRAIN_END))
training_reference = featured.loc[reference_mask]
training_p95 = float(training_reference["next_week_change_pct"].quantile(.95))


## 7. Inspect the largest training increases
 Check both weeks' ranges, price flags, coverage and the following training week. Large changes are not automatically errors; unchanged premise counts do not prove unchanged contributors.


In [ ]:
# All prices used for this inspection must belong to training weeks.
training_weeks = weekly.loc[weekly["week_end"] <= TRAIN_END]
audit_fields = ["weekly_min_price", "weekly_max_price", "record_count", "premise_count",
                "observed_days", "suspicious_record_count"]
next_details = training_weeks[SERIES_KEYS + ["week_start"] + audit_fields].rename(
    columns={"week_start": "next_week_start", **{c: "target_" + c for c in audit_fields}})
largest = training_reference.nlargest(20, "next_week_change_pct").merge(
    next_details, on=SERIES_KEYS + ["next_week_start"], how="left", validate="one_to_one")
largest["after_target_week_start"] = largest["next_week_start"] + pd.Timedelta(days=7)
after_details = training_weeks[SERIES_KEYS + ["week_start", "weekly_median_price"]].rename(
    columns={"week_start": "after_target_week_start", "weekly_median_price": "after_target_price"})
largest = largest.merge(after_details, on=SERIES_KEYS + ["after_target_week_start"], how="left", validate="one_to_one")
review_columns = SERIES_KEYS + ["week_start", "next_week_start", "weekly_median_price", "next_week_price",
    "next_week_change_pct", "after_target_price"] + audit_fields + ["target_" + c for c in audit_fields]
extreme_review = largest[review_columns].copy()
extreme_review.insert(0, "case_id", range(len(extreme_review)))
extreme_review["premise_count_changed"] = extreme_review["premise_count"].ne(extreme_review["target_premise_count"])
extreme_review["either_week_flagged"] = (extreme_review["suspicious_record_count"] > 0) | (extreme_review["target_suspicious_record_count"] > 0)
extreme_review["review_status"] = "diagnostic only: cause not automatically established"
display(extreme_review[["case_id", "item", "state", "district", "week_start", "weekly_median_price",
    "next_week_price", "next_week_change_pct", "premise_count_changed", "either_week_flagged"]].head(10))


## 8. Check contributing premises and save daily evidence
 For each case, compare premise identities across the two weeks. The common-premise comparison takes a median within each premise/week, then a median across matched premises, giving matched premises equal weight. It is a diagnostic and does not replace the observation-weighted label.
 Review units, actual daily prices, promotions and monitoring frequency. Cause remains unresolved until supported by source evidence.


In [ ]:
daily_review_available = cfg.STAGE3_MODEL_READY_PARQUET.exists()
if daily_review_available:
    daily = pd.read_parquet(cfg.STAGE3_MODEL_READY_PARQUET,
                            columns=SERIES_KEYS + ["date", "premise_code", "price", "is_suspicious_price"])
    daily["date"] = pd.to_datetime(daily["date"])
    daily = daily.loc[daily["date"] <= TRAIN_END].copy()
    daily["week_start"] = daily["date"] - pd.to_timedelta(daily["date"].dt.dayofweek, unit="D")
    source_rows, panel_rows = [], []
    for _, case in extreme_review.iterrows():
        pair = daily.loc[daily["item_code"].eq(case["item_code"]) &
                         daily["week_start"].isin([case["week_start"], case["next_week_start"]])].copy()
        for key in SERIES_KEYS:
            pair = pair.loc[pair[key].eq(case[key])]
        pair["case_id"] = case["case_id"]
        source_rows.append(pair)
        panel = pair.groupby(["premise_code", "week_start"])["price"].median().unstack("week_start")
        current = panel.get(case["week_start"], pd.Series(dtype=float)).dropna()
        target = panel.get(case["next_week_start"], pd.Series(dtype=float)).dropna()
        common = current.index.intersection(target.index)
        union = current.index.union(target.index)
        current_price, target_price = current.loc[common].median(), target.loc[common].median()
        panel_rows.append({"case_id": case["case_id"], "common_premises": len(common),
            "premise_overlap_pct": 100 * len(common) / len(union) if len(union) else np.nan,
            "common_panel_change_pct": (target_price / current_price - 1) * 100 if current_price > 0 else np.nan})
    pd.concat(source_rows, ignore_index=True).to_csv(OUTPUT_DIR / "extreme_change_daily_records.csv", index=False)
    extreme_review = extreme_review.merge(pd.DataFrame(panel_rows), on="case_id", validate="one_to_one")
    display(extreme_review[["case_id", "common_premises", "premise_overlap_pct", "common_panel_change_pct"]].head(10))
    del daily, source_rows
else:
    print("Daily file unavailable: causes cannot be confirmed from weekly aggregates alone.")

extreme_review.to_csv(OUTPUT_DIR / "extreme_training_changes.csv", index=False)


## 9a. Check the documented-record decision on training data only
Restore the documented records in memory and recalculate affected weekly summaries/features. Compare eligible and model-ready training pairs using identical coverage, history and fixed thresholds. Validation/test results do not select the cleaning policy. This checks label sensitivity; it does not prove all remaining source records are valid or model scores robust.


In [ ]:
cleaning = json.loads((cfg.STAGE3_DIR / "cleaning_summary.json").read_text())
if not cleaning["reviewed_daily_records"] or cfg.REMOVE_FLAGGED_PRICES:
    print("Sensitivity skipped: no documented exclusions, or optional statistical removal is enabled.")
else:
    restored = pd.DataFrame(cleaning["reviewed_daily_records"])
    restored["date"] = pd.to_datetime(restored["date"])
    restored["week_start"] = restored["date"] - pd.to_timedelta(restored["date"].dt.dayofweek, unit="D")
    affected_keys = restored[SERIES_KEYS].drop_duplicates()
    daily = pd.read_parquet(cfg.STAGE3_MODEL_READY_PARQUET)
    daily["date"] = pd.to_datetime(daily["date"])
    daily["week_start"] = daily["date"] - pd.to_timedelta(daily["date"].dt.dayofweek, unit="D")
    affected = daily.merge(affected_keys, on=SERIES_KEYS, how="inner")
    affected = pd.concat([affected, restored], ignore_index=True)
    raw_weekly = affected.groupby(SERIES_KEYS + ["week_start"]).agg(
        weekly_median_price=("price", "median"), record_count=("price", "size"),
        premise_count=("premise_code", "nunique"), observed_days=("date", "nunique"),
        weekly_std_price=("price", "std"), weekly_min_price=("price", "min"),
        weekly_max_price=("price", "max"), suspicious_record_count=("is_suspicious_price", "sum")
    ).reset_index()
    raw_weekly = raw_weekly.loc[raw_weekly["week_start"].between(weekly["week_start"].min(), weekly["week_start"].max())]
    raw_parts = [make_features(part) for _, part in raw_weekly.groupby(SERIES_KEYS, sort=False)]
    raw_features = pd.concat(raw_parts, ignore_index=True)
    raw_features["next_week_end"] = raw_features["week_start"] + pd.Timedelta(days=13)
    marked = featured.merge(affected_keys.assign(affected=True), on=SERIES_KEYS, how="left", validate="many_to_one")
    with_six = pd.concat([marked.loc[marked["affected"].ne(True)], raw_features], ignore_index=True)
    rows = []
    for policy, frame in [("documented_excluded", featured), ("documented_retained", with_six)]:
        eligible = ((frame["record_count"] >= MIN_RECORDS) & (frame["premise_count"] >= MIN_PREMISES)
            & (frame["next_week_record_count"] >= MIN_RECORDS) & (frame["next_week_premise_count"] >= MIN_PREMISES)
            & frame["next_week_price"].notna() & (frame["next_week_end"] <= TRAIN_END))
        history = frame["lag_1w_price"].notna() & (frame["history_count_4w"] >= MIN_HISTORY_4W)
        for population, mask in [("eligible_pairs", eligible), ("model_ready", eligible & history)]:
            changes = frame.loc[mask, "next_week_change_pct"]
            row = {"policy": policy, "population": population, "rows": len(changes),
                   "p95_pct": changes.quantile(0.95), "max_change_pct": changes.max()}
            for cutoff in [5.0, 10.0, 15.0, max(5.0, training_p95)]:
                row[f"surges_at_{cutoff:.4f}pct"] = int(changes.ge(cutoff).sum())
            rows.append(row)
    source_sensitivity = pd.DataFrame(rows)
    display(source_sensitivity)
    source_sensitivity.to_csv(OUTPUT_DIR / "training_source_sensitivity.csv", index=False)
    del daily, affected, restored, marked, with_six, raw_features, raw_parts
